In [21]:
import os
import time
import json
import numpy as np
import pandas as pd
import pm4py
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import optuna
import warnings
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate
from tensorflow.keras.optimizers import Adam
from ACF_code import activity_contex_frequency, pmi
from TAX_LSTM import next_activity_prediction_tax


ModuleNotFoundError: No module named 'definitions'

In [ ]:
# Importing dataset from file path
def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    return pm4py.convert_to_dataframe(log)

# Cleaning dataset: removing unnecessary columns, shifting to resource focus
def clean_dataset(df):
    df_final = df[['case:concept:name', 'concept:name', 'org:resource', 'time:timestamp']]
    df_final = df_final.sort_values(['org:resource', 'time:timestamp'])
    return df_final


# creating 80/20 split based on resources, ensuring a resource is in EITHER the test set OR the train set
def create_split(df_clean, test_size):
    resource_traces = (
        df_clean.sort_values(["org:resource", "time:timestamp"])
               .groupby("org:resource")["concept:name"]
               .apply(list)
    )

    resource_traces = resource_traces[resource_traces.apply(len) > 1]

    resources = resource_traces.index.tolist()

    # create set of train/test resource ids
    train_res, test_res = train_test_split(
        resources,
        test_size=test_size,
        random_state=1
    )

    train_traces = resource_traces.loc[train_res]
    test_traces = resource_traces.loc[test_res]

    return train_traces, test_traces


# prefix extraction on set list of prefix lengths, already implicitly buckets on prefix length
def build_prefix_df(resource_traces, prefix_lengths=[10], sliding_window=False):
    all_rows = []
    for length in prefix_lengths:
        for resource, seq in resource_traces.items():

            if(len(seq) < length + 1):
                continue

            if(sliding_window):
                for i in range(length, len(seq)):
                    prefix = seq[i-length:i]
                    next_act = seq[i]

                    all_rows.append({
                    'resource': resource,
                    'subtrace': prefix,
                    'prefix_length': length,
                    'last_activity': prefix[-1],
                    'next_activity': next_act
                    })
            else:
                prefix = seq[-(length+1):-1]
                next_act = seq[-1]

                all_rows.append({
                'resource': resource,
                'subtrace': prefix,
                'prefix_length': length,
                'last_activity': prefix[-1],
                'next_activity': next_act
                })
            
    return pd.DataFrame(all_rows)




def process_dataset(df, prefix_lengths):
    df_clean = clean_dataset(df)

    train_split, test_split = create_split(df_clean, 0.2)

    train_df = build_prefix_df(train_split, prefix_lengths, False)
    test_df = build_prefix_df(test_split, prefix_lengths, False)

    return train_df, test_df, train_split, test_split


In [4]:
# df_2013, df_2013_prefixes = process_dataset("datasets/BPI_Challenge_2013_incidents.xes")
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")
#train_df_2013, test_df_2013 = process_dataset(df_2013)
GLOBAL_prefix_lengths = [10, 20, 30, 40, 50, 75, 100, 125, 150]
#GLOBAL_prefix_lengths = [100, 150, 200, 300, 400, 500, 600, 700, 800]

train_df_2013, test_df_2013 = process_dataset(df_2013, GLOBAL_prefix_lengths)

/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/pm4py/utils.py:992: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:02<00:00, 3319.05it/s]
